# 01 — Data Loading

This notebook loads and inspects the two core datasets from the [DepMap portal](https://depmap.org/portal/) that power this project.

> **Files were downloaded manually** from [depmap.org/portal/data_page](https://depmap.org/portal/data_page/) and placed in `data/raw/`. See the manual download instructions at the bottom of this notebook if you need to re-download them.

---

## Why CCLE / DepMap?

The **Cancer Cell Line Encyclopedia (CCLE)**, distributed through the Broad Institute's DepMap portal, is the primary data source for this project for several reasons:

| Reason | Detail |
|--------|--------|
| **Scale** | ~1,800+ cancer cell lines across >30 tissue lineages |
| **Data alignment** | RNA-seq expression and mutation calls are derived from the *same* cell line cultures — no cross-cohort batch effects |
| **Curation quality** | Mutations are called with a standardised pipeline (GATK/MuTect2), then filtered for germline artefacts and manually reviewed for key cancer genes |
| **Public access** | Freely available; no data-use agreement required for the files used here |
| **TP53 coverage** | TP53 is one of the most frequently mutated genes in CCLE (~40–60% of lines depending on lineage), giving sufficient positive examples for classification |

---

## Files we use

### 1. `OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv` — gene expression

- **What it is**: RNA-seq expression values for all human protein-coding genes, normalised to **log₂(TPM + 1)**. The `+1` pseudo-count prevents log(0) and compresses the dynamic range of lowly expressed genes.
- **Shape**: `(cell lines) × (genes)` — roughly 1,450 rows × 19,000 columns.
- **Index column**: `DepMap_ID` — a stable identifier like `ACH-000001`.
- **Gene columns**: formatted as `SYMBOL (ENTREZ_ID)`, e.g. `TP53 (7157)`. We will strip the Entrez suffix during preprocessing.
- **This is our feature matrix X** for every downstream model.

### 2. `OmicsSomaticMutations.csv` — somatic mutation calls

- **What it is**: One row per mutation event (variant-level, not cell-line-level). Aggregating by `ModelID` + `HugoSymbol == 'TP53'` gives us our target labels.
- **Key columns we expect**:

| Column | Description |
|--------|-------------|
| `ModelID` | DepMap cell line identifier (= `DepMap_ID`) |
| `HugoSymbol` | Gene name (we filter for `TP53`) |
| `VariantClassification` | Mutation consequence: `Missense_Mutation`, `Nonsense_Mutation`, `Frame_Shift_Del/Ins`, `Splice_Site`, etc. |
| `VariantType` | `SNP`, `DEL`, `INS` |
| `Chromosome`, `Start_Pos`, `End_Pos` | Genomic coordinates |
| `ReferenceAllele`, `AltAllele` | Alleles |
| `ProteinChange` | HGVS protein-level notation, e.g. `p.R175H` |
| `isCOSMIChotspot` | Boolean — whether this variant is a COSMIC hotspot |

- From this file we will derive **binary labels** (TP53 mutant vs. wild-type) and **multi-class labels** (missense / nonsense / frameshift / splice / WT) for Tasks 1 and 2.

---

## Setup

In [ ]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")

EXPR_FILE = RAW_DIR / "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"
MUT_FILE  = RAW_DIR / "OmicsSomaticMutations.csv"

for f in (EXPR_FILE, MUT_FILE):
    status = "OK" if f.exists() else "MISSING"
    print(f"[{status}] {f.name}")

---

## Inspect: Gene Expression

In [ ]:
expr = pd.read_csv(EXPR_FILE, index_col=0)

print("Shape:", expr.shape)
print(f"  {expr.shape[0]:,} cell lines × {expr.shape[1]:,} genes")
print()
print("Index name:", expr.index.name)
print("First 5 index values:", expr.index[:5].tolist())
print()
print("First 5 gene column names:", expr.columns[:5].tolist())

In [ ]:
expr.iloc[:5, :8]

In [ ]:
# Sanity-check value range — log2(TPM+1) should be in roughly [0, 20]
print("Value range:")
print(expr.stack().describe())

In [ ]:
# Check that TP53 is present (column name format: 'TP53 (7157)')
tp53_cols = [c for c in expr.columns if c.startswith("TP53 ")]
print("TP53 expression column(s):", tp53_cols)

if tp53_cols:
    print()
    print(expr[tp53_cols[0]].describe())

---

## Inspect: Somatic Mutations

In [ ]:
mut = pd.read_csv(MUT_FILE, low_memory=False)

print("Shape:", mut.shape)
print(f"  {mut.shape[0]:,} mutation events across {mut['ModelID'].nunique():,} cell lines")
print()
print("Columns:")
print(mut.columns.tolist())

In [ ]:
mut.head(5)

In [ ]:
# TP53 mutations only
tp53_mut = mut[mut["HugoSymbol"] == "TP53"].copy()

print(f"TP53 mutation events : {len(tp53_mut):,}")
print(f"Cell lines with TP53 mutation : {tp53_mut['ModelID'].nunique():,}")
print()
print("VariantClassification breakdown:")
print(tp53_mut["VariantClassification"].value_counts())

In [ ]:
tp53_mut[["ModelID", "HugoSymbol", "VariantClassification",
           "VariantType", "ProteinChange", "isCOSMIChotspot"]].head(10)

---

## Summary

| File | Rows | Columns | Notes |
|------|------|---------|-------|
| Expression | ~1,450 (cell lines) | ~19,000 (genes) | log₂(TPM+1); index = `DepMap_ID` |
| Mutations | ~1.8M (events) | ~30+ | One row per variant; filter on `HugoSymbol == 'TP53'` for labels |

Both files are in `data/raw/`. The next notebook (`02_preprocessing.ipynb`) will:
- Align cell lines present in both datasets
- Derive binary and multi-class TP53 labels
- Handle class imbalance and split into train/test sets

---

## Manual download instructions

If you need to re-download these files:

1. Go to **[depmap.org/portal/data_page](https://depmap.org/portal/data_page/)**
2. Select the desired release from the **Release** dropdown.
3. Search for and download:
   - `OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv`
   - `OmicsSomaticMutations.csv`
4. Place both files in `data/raw/` and re-run this notebook.